In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers==4.45.0 accelerate==0.27.2
!pip install -q scipy pillow opencv-python
!pip install -q nltk rouge-score

print("All libraries installed!")

In [ ]:
# Cell 2: Verify GPU
import torch

print("=" * 60)
print("SYSTEM CHECK")
print("=" * 60)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("NO GPU! Go back to Settings and enable GPU T4")
    
print("=" * 60)


In [ ]:
# Cell 3: Setup - Copy files to working directory
import sys
import os
from pathlib import Path
import shutil

print("Setting up project files...\n")

# Source path (read-only dataset)
DATASET_PATH = "/kaggle/input/xai-chest-x-ray-project-final-year/xai-project/kaggle-upload"

# Working directory (writable)
WORK_DIR = "/kaggle/working/"

# Copy src_v2 to working directory
src_source = Path(f"{DATASET_PATH}/src_v2")
src_dest = Path(f"{WORK_DIR}/src_v2")

print(f"Copying src_v2 folder to working directory...")

# Remove if exists (in case you're re-running)
if src_dest.exists():
    shutil.rmtree(src_dest)

# Copy the entire folder
shutil.copytree(src_source, src_dest)

# Add to Python path
sys.path.insert(0, WORK_DIR)

print(f"Code copied to: {src_dest}")
print(f"   Files copied: {len(list(src_dest.glob('*.py')))} Python files")

# Fix relative imports in the copied files
print(f"\n🔧 Fixing relative imports...")

for py_file in src_dest.glob("*.py"):
    content = py_file.read_text()
    # Replace relative imports with absolute imports
    content = content.replace("from .config import", "from src_v2.config import")
    content = content.replace("from .xai_enhanced import", "from src_v2.xai_enhanced import")
    content = content.replace("from .behavior_extractor import", "from src_v2.behavior_extractor import")
    content = content.replace("from .llm_explainer import", "from src_v2.llm_explainer import")
    content = content.replace("from .prompt_utils import", "from src_v2.prompt_utils import")
    content = content.replace("from .visualization import", "from src_v2.visualization import")
    
    # Write back
    py_file.write_text(content)

print(f"Imports fixed!")

# Verify other files
model_path = Path(f"{DATASET_PATH}/models/denseNet121_v2.pth")
sample_path = Path(f"{DATASET_PATH}/Test-Samples/sample-2.jpg")

if model_path.exists():
    print(f"\nModel found: {model_path.name}")
    print(f"   Size: {model_path.stat().st_size / 1024**2:.1f} MB")
else:
    print(f"\nModel NOT found at {model_path}")

if sample_path.exists():
    print(f"\nTest image found: {sample_path.name}")
else:
    print(f"\nTest image NOT found at {sample_path}")

# Store paths for later use
MODEL_PATH = str(model_path)
SAMPLE_PATH = str(sample_path)

print("\n" + "=" * 60)
print("Setup complete! Ready to import modules.")
print("=" * 60)


In [ ]:
# Cell 5: Import your modules
print("Loading your code modules...\n")

from src_v2.config import *
from src_v2.xai_enhanced import EnhancedGradCAM
from src_v2.behavior_extractor import ModelBehaviorExtractor

print("Config loaded")
print("Grad-CAM module loaded")
print("Behavior Extractor loaded")

# We'll load LLM separately in next cell
print("\nCore modules ready!")


In [ ]:
# Cell 6.5: Fix gaussian_filter compatibility issue
import sys

# Read the xai_enhanced.py file
xai_file = Path("/kaggle/working/src_v2/xai_enhanced.py")
content = xai_file.read_text()

# Find and replace the gaussian_filter section
old_code = """        # ADD SMOOTHING HERE
        from scipy.ndimage import gaussian_filter
        cam = gaussian_filter(cam, sigma=2)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # Re-normalize"""

new_code = """        # ADD SMOOTHING HERE (with dtype fix for scipy)
        from scipy.ndimage import gaussian_filter
        # Convert to float32 for scipy compatibility
        cam = cam.astype(np.float32)
        cam = gaussian_filter(cam, sigma=2)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # Re-normalize"""  

content = content.replace(old_code, new_code)

# Write back
xai_file.write_text(content)

print("Fixed gaussian_filter compatibility issue!")
print("Now supports GPU float16 operations")


In [ ]:
# Cell 6.6: Reload the fixed module
import importlib
import sys

# Remove cached module
if 'src_v2.xai_enhanced' in sys.modules:
    del sys.modules['src_v2.xai_enhanced']

# Re-import
from src_v2.xai_enhanced import EnhancedGradCAM

print("Module reloaded with fix!")


In [ ]:
import torch.nn as nn
from torchvision import models

print("Rebuilding your DenseNet-121 architecture...")

class ChestXRayModel(nn.Module):
    def __init__(self, num_classes=14):
        super(ChestXRayModel, self).__init__()
        self.backbone = models.densenet121(weights=None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = ChestXRayModel(num_classes=14)
checkpoint = torch.load(MODEL_PATH, map_location='cuda', weights_only=False)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
    print(f"  Checkpoint from epoch {checkpoint.get('epoch', '?')}")
else:
    state_dict = checkpoint

state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=True)  # strict=True confirms perfect match
model.eval()
model.to('cuda')
print(f"DenseNet-121 is ready!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("Done!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 1: COMPUTER VISION] — Load Image & Predict
# ═══════════════════════════════════════════════════════════════
from PIL import Image
from torchvision import transforms
import numpy as np
import torch

print("=" * 70)
print("COMPONENT 1: COMPUTER VISION — DIAGNOSTIC PREDICTION")
print("=" * 70)

# Load test image
img = Image.open(SAMPLE_PATH).convert('RGB')
print(f"\nImage loaded: {img.size}")

# Preprocess
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_tensor = transform(img).unsqueeze(0).to('cuda')

# Predict
with torch.no_grad():
    logits = model(img_tensor)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

LABEL_NAMES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
    "Pneumonia", "Pneumothorax"
]

top_indices = np.argsort(probs)[::-1][:5]
print("\nTop 5 Predictions:")
for idx in top_indices:
    status = "✓" if probs[idx] > 0.5 else "○"
    print(f"  {status} {LABEL_NAMES[idx]:20s}: {probs[idx]*100:5.1f}%")

top_disease_idx = top_indices[0]
top_disease = LABEL_NAMES[top_disease_idx]
top_confidence = probs[top_disease_idx]

print(f"\nPrimary Finding: {top_disease} ({top_confidence*100:.1f}%)")
print("=" * 70)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 2: XAI] — Grad-CAM & Behavioral Analysis
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("COMPONENT 2: XAI — GRAD-CAM & BEHAVIOR EXTRACTION")
print("=" * 70)

# --- Grad-CAM ---
print("\nGenerating Grad-CAM heatmap...")
# gradcam = EnhancedGradCAM(model, target_layer="backbone.layer4")
gradcam = EnhancedGradCAM(model, target_layer='backbone.features.denseblock4')
heatmap = gradcam.generate_cam(img_tensor, top_disease_idx, device='cuda')

print(f"  Heatmap shape : {heatmap.shape}")
print(f"  Peak attention  : {heatmap.max():.3f}")
print(f"  Mean attention  : {heatmap.mean():.3f}")

# --- Behavior Extraction ---
print("\nExtracting model behavior...")
behavior_extractor = ModelBehaviorExtractor()
behavior = behavior_extractor.extract_complete_behavior(
    predictions={top_disease: float(top_confidence)},
    heatmap=heatmap,
    all_probabilities=probs
)

spatial = behavior["spatial_analysis"]
regions = behavior["anatomical_regions"]

print(f"  Coverage     : {spatial['attention_percentage']:.1f}% of lung area")
print(f"  Pattern         : {'Focal' if spatial['is_focal'] else 'Diffuse'}")
print(f"  Peak location   : ({spatial['peak_location']['x']}, {spatial['peak_location']['y']})")

if regions:
    print(f"\n  Anatomical regions (top 3):")
    for r in regions[:3]:
        print(f"    • {r['name'].replace('_',' ').title()}: {r['attention_score']*100:.0f}%")

# Store for visualization
results = {
    'image': img,
    'heatmap': heatmap,
    'disease': top_disease,
    'confidence': top_confidence,
    'behavior': behavior,
    'probs': probs
}

print("\n" + "=" * 70)
print("XAI Complete!")
print("=" * 70)


In [ ]:
import cv2, os
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines   as mlines
from matplotlib.patches   import FancyBboxPatch
from matplotlib.gridspec  import GridSpec
import numpy as np

# ══════════════════════════════════════════════════════════════
# TRICK 3 — Global rcParams for sharpest rendering
# ══════════════════════════════════════════════════════════════
mpl.rcParams.update({
    'text.antialiased':    True,
    'lines.antialiased':   True,
    'patch.antialiased':   True,
    'image.interpolation': 'lanczos',
    'image.resample':      True,
    'figure.autolayout':   False,
    'savefig.dpi':         300,
    'savefig.bbox':        'tight',
    'savefig.pad_inches':  0.05,
    'font.family':         'DejaVu Sans',
})

print("Generating 300 DPI thesis-quality visualizations...")

# ── High-res base images ───────────────────────────────────────
SIZE = 600
img_array = np.array(results['image'].resize((SIZE, SIZE)))
if len(img_array.shape) == 2:
    img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)

heatmap_res = cv2.resize(results['heatmap'], (SIZE, SIZE), interpolation=cv2.INTER_CUBIC)
heatmap_res = np.clip(heatmap_res, 0.0, 1.0)          # kills blue dot artefact

heatmap_col = cv2.applyColorMap((heatmap_res * 255).astype(np.uint8), cv2.COLORMAP_JET)
heatmap_rgb = cv2.cvtColor(heatmap_col, cv2.COLOR_BGR2RGB)
overlay     = cv2.addWeighted(img_array, 0.55, heatmap_rgb, 0.45, 0)

binary_mask = (heatmap_res > 0.6).astype(np.uint8) * 255
contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

def compute_region_attention(h):
    H, W = h.shape; r3, c3 = H // 3, W // 2
    return {
        'Upper-R': float(h[:r3,      c3:].mean()),
        'Upper-L': float(h[:r3,     :c3].mean()),
        'Mid-R':   float(h[r3:2*r3,  c3:].mean()),
        'Mid-L':   float(h[r3:2*r3, :c3].mean()),
        'Lower-R': float(h[2*r3:,    c3:].mean()),
        'Lower-L': float(h[2*r3:,   :c3].mean()),
        'Central': float(h[r3:2*r3,    :].mean()),
    }

rs = compute_region_attention(results['heatmap'])
def attn_hex(s):  return '#FF6B6B' if s > 0.7 else '#FFB347' if s > 0.3 else '#C9B1FF'
def attn_rgba(s): return (1.0,0.42,0.42,.22) if s>.7 else (1.0,.70,.28,.22) if s>.3 else (.79,.69,1.0,.22)

# helper — save both PNG + PDF
def save_both(path_no_ext, fig, facecolor='white'):
    fig.savefig(f'{path_no_ext}.png', dpi=300, facecolor=facecolor, format='png')
    fig.savefig(f'{path_no_ext}.pdf',           facecolor=facecolor, format='pdf')  # vector — use this in Word
    kb = os.path.getsize(f'{path_no_ext}.png') // 1024
    print(f"Saved → {os.path.basename(path_no_ext)}.png ({kb} KB)  +  .pdf (vector)")

# ══════════════════════════════════════════════════════════════
# VIZ 1 — Original | Pure Heatmap | AI Focus Areas
# ══════════════════════════════════════════════════════════════
# TRICK 2 — layout='constrained' replaces tight_layout entirely
fig1, axs = plt.subplots(1, 3, figsize=(7, 2.8), dpi=300, layout='constrained')
fig1.patch.set_facecolor('white')

axs[0].imshow(img_array,   interpolation='lanczos')
axs[0].set_title('Original X-Ray',   fontsize=10, fontweight='bold'); axs[0].axis('off')

axs[1].imshow(heatmap_res, cmap='jet', interpolation='lanczos')
axs[1].set_title('Grad-CAM Heatmap', fontsize=10, fontweight='bold'); axs[1].axis('off')

axs[2].imshow(overlay,     interpolation='lanczos')
for cnt in contours:
    if cv2.contourArea(cnt) > 800:
        bx, by, bw, bh = cv2.boundingRect(cnt)
        axs[2].add_patch(patches.Rectangle(
            (bx, by), bw, bh, lw=2, edgecolor='#00FF00', facecolor='none'))
axs[2].set_title(f'AI Focus Areas\n{results["disease"]} ({results["confidence"]*100:.1f}%)',
                 fontsize=10, fontweight='bold'); axs[2].axis('off')

save_both('/kaggle/working/xai_result_300dpi', fig1, facecolor='white')
plt.show()

# ══════════════════════════════════════════════════════════════
# VIZ 2 — Original | Grad-CAM Overlay | Anatomy Map (dark)
# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
# FIXED draw_anatomy_map — uses 10×10 coords (matches panel shape)
# ══════════════════════════════════════════════════════════════
def draw_anatomy_map(ax, rs):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)               # ← was ylim(0,15) — caused vertical squish
    ax.set_facecolor('#0d1117')
    ax.axis('off')

    # Outer dashed border
    ax.add_patch(FancyBboxPatch((0.2, 0.2), 9.6, 9.6,
                 boxstyle='square,pad=0', lw=1.2,
                 edgecolor='#58a6ff', facecolor='none', linestyle='--'))

    # Title — properly centered inside border
    ax.text(5, 9.6, 'LUNG  ANATOMY  MAP', ha='center', va='center',
            color='white', fontsize=9, fontweight='bold', family='monospace')

    # 6 region boxes — evenly spaced in 10×10 space
    layout = [
        (0.3, 6.7, 4.4, 2.5, 'UPPER-R', 'Upper-R'),
        (5.3, 6.7, 4.4, 2.5, 'UPPER-L', 'Upper-L'),
        (0.3, 3.9, 4.4, 2.5, 'MID-R',   'Mid-R'),
        (5.3, 3.9, 4.4, 2.5, 'MID-L',   'Mid-L'),
        (0.3, 2.5, 4.4, 1.2, 'LOWER-R', 'Lower-R'),
        (5.3, 2.5, 4.4, 1.2, 'LOWER-L', 'Lower-L'),
    ]
    for bx, by, bw, bh, lbl, key in layout:
        s = rs.get(key, 0)
        ax.add_patch(FancyBboxPatch((bx, by), bw, bh,
                     boxstyle='round,pad=0.05', lw=0.8,
                     edgecolor='#4a5568', facecolor='#161b22'))
        # Label — anchored relative to box top
        ax.text(bx+bw/2, by+bh-0.28, lbl, ha='center', va='center',
                color='white', fontsize=7, fontweight='bold', family='monospace')
        # Dot + percentage — anchored in box center
        dot_y = by + bh/2 - 0.1
        ax.plot(bx+bw/2, dot_y, 'o', color=attn_hex(s), markersize=16, zorder=5)
        ax.text(bx+bw/2, dot_y, f'{s:.0%}', ha='center', va='center',
                color='black', fontsize=6, fontweight='bold', zorder=6)

    # Central zone bar
    c = rs.get('Central', 0)
    ax.add_patch(FancyBboxPatch((0.3, 1.85), 9.4, 0.50,
                 boxstyle='round,pad=0.05', lw=0.8,
                 edgecolor='#4a5568', facecolor='#1a1f2e'))
    ax.plot(1.1, 2.10, 'o', color=attn_hex(c), markersize=8, zorder=5)
    ax.text(5, 2.10, f' CENTRAL ZONE  {c:.0%} ', ha='center', va='center',
            color='white', fontsize=6.5, fontweight='bold', family='monospace')

    # Legend — evenly spaced with 0.40 unit gaps
    for i, (col, txt) in enumerate([('#FF6B6B', 'High attention  (>0.70)'),
                                     ('#FFB347', 'Moderate  (0.30–0.70)'),
                                     ('#C9B1FF', 'Low attention  (<0.30)')]):
        yp = 1.50 - i * 0.40
        ax.plot(0.6, yp, 'o', color=col, markersize=7, zorder=5)
        ax.text(1.1, yp, txt, va='center', color='#8b949e',
                fontsize=5.5, family='monospace')

# ══════════════════════════════════════════════════════════════
# VIZ 2 — updated figsize + GridSpec to make anatomy panel square
# ══════════════════════════════════════════════════════════════
fig2 = plt.figure(figsize=(9, 4.0), dpi=300, layout='constrained')  # height 3.5→4.0
fig2.patch.set_facecolor('#0d1117')
gs = GridSpec(1, 3, figure=fig2, width_ratios=[1, 1, 1.35])          # was 1.25

ax2a = fig2.add_subplot(gs[0])
ax2a.imshow(img_array, interpolation='lanczos')
ax2a.set_title('Original X-Ray', color='white', fontsize=10, fontweight='bold', pad=6)
ax2a.axis('off'); ax2a.set_facecolor('#0d1117')

ax2b = fig2.add_subplot(gs[1])
ax2b.imshow(overlay, interpolation='lanczos')
ax2b.set_title(f'Grad-CAM Overlay\n{results["disease"]} ({results["confidence"]*100:.1f}%)',
               color='white', fontsize=10, fontweight='bold', pad=6)
ax2b.axis('off'); ax2b.set_facecolor('#0d1117')

ax2c = fig2.add_subplot(gs[2])
draw_anatomy_map(ax2c, rs)

fig2.suptitle(f'XAI Analysis — DenseNet-121  |  Finding: {results["disease"]} ({results["confidence"]*100:.1f}%)',
              color='white', fontsize=11, fontweight='bold')

save_both('/kaggle/working/xai_analysis_3panel_300dpi', fig2, facecolor='#0d1117')
plt.show()


# ══════════════════════════════════════════════════════════════
# VIZ 3 — Grad-CAM + Anatomical Region Map (standalone)
# ══════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(5, 5), dpi=300, layout='constrained')
fig3.patch.set_facecolor('#0d1117')
ax3.imshow(overlay, interpolation='lanczos', extent=[0,SIZE,SIZE,0])
ax3.set_xlim(0,SIZE); ax3.set_ylim(SIZE,0); ax3.axis('off')

H3, C3 = SIZE//3, SIZE//2
grid = [(0,   0,  C3,H3,'Upper-L','Upper-L'),(C3,  0, C3,H3,'Upper-R','Upper-R'),
        (0,  H3,  C3,H3,'Mid-L',  'Mid-L'),  (C3, H3, C3,H3,'Mid-R',  'Mid-R'),
        (0, 2*H3, C3,H3,'Lower-L','Lower-L'),(C3,2*H3,C3,H3,'Lower-R','Lower-R')]

for gx,gy,gw,gh,lbl,key in grid:
    s = rs.get(key, 0)
    ax3.add_patch(patches.Rectangle((gx,gy),gw,gh, facecolor=attn_rgba(s), linewidth=0))
    ax3.add_patch(patches.Rectangle((gx,gy),gw,gh, facecolor='none',
                  linewidth=1.5, edgecolor=attn_hex(s), linestyle='--'))
    ax3.text(gx+gw/2, gy+22, lbl, ha='center', va='center',
             color='white', fontsize=9, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.65, lw=0))
    ax3.text(gx+gw/2, gy+gh/2, f'{s:.0%}', ha='center', va='center',
             color='white', fontsize=14, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=attn_hex(s), alpha=0.85, lw=0))

c = rs.get('Central', 0)
ax3.add_patch(patches.Rectangle((55,SIZE-48),SIZE-110,36,
              facecolor='black', alpha=0.75, lw=1.2, edgecolor=attn_hex(c)))
ax3.text(SIZE/2, SIZE-30, f'CENTRAL ZONE  {c:.0%}',
         ha='center', va='center', color='white', fontsize=9, fontweight='bold')

h1 = mlines.Line2D([],[],color='#FF6B6B',marker='o',ls='None',ms=9,label='High  (>70%)')
h2 = mlines.Line2D([],[],color='#FFB347',marker='o',ls='None',ms=9,label='Moderate  (30–70%)')
h3 = mlines.Line2D([],[],color='#C9B1FF',marker='o',ls='None',ms=9,label='Low  (<30%)')
ax3.legend(handles=[h1,h2,h3], loc='lower right', fontsize=8,
           facecolor='#0d1117', edgecolor='#58a6ff', labelcolor='white', framealpha=0.88)
ax3.set_title(f'Grad-CAM + Anatomical Region Map\n{results["disease"]} ({results["confidence"]*100:.1f}%)',
              color='white', fontsize=10, fontweight='bold', pad=10)

save_both('/kaggle/working/xai_region_overlay_300dpi', fig3, facecolor='#0d1117')
plt.show()

print("\n✔  ALL 3 VISUALIZATIONS COMPLETE (PNG + PDF)")
print("Use the .pdf files in Word for perfect quality at any zoom!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DOWNLOAD VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════
import os
from IPython.display import FileLink, display

viz_files = [
    ('xai_result.png',          'VIZ 1 — Original | Heatmap | BBox Overlay'),
    ('xai_analysis_3panel.png', 'VIZ 2 — Original | Grad-CAM | Anatomy Map'),
    ('xai_region_overlay.png',  'VIZ 3 — Grad-CAM with Region Grid Overlay'),
]

print("DOWNLOAD LINKS")
print("=" * 55)
for fname, label in viz_files:
    fpath = f'/kaggle/working/{fname}'
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"\n  {label}")
        print(f"  Size: {size_kb:.1f} KB")
        display(FileLink(fname))
    else:
        print(f"\n  Not found: {fname} — run the viz cell first")

print("\n" + "=" * 55)
print("TIP: Also available in Kaggle Output tab (right panel)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 3: NLP] — Load Qwen2.5-3B-Instruct
# ═══════════════════════════════════════════════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

QWEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("=" * 70)
print("COMPONENT 3: NLP — LOADING QWEN2.5-3B-INSTRUCT")
print("=" * 70)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)
print("Tokenizer ready")

print("Loading model in float16...")
llm_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
llm_model.eval()

print(f"\nQwen2.5-3B loaded!")
print(f"   GPU Memory used : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("=" * 70)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 3: NLP] — Qwen2.5 Explanation Generator
# ═══════════════════════════════════════════════════════════════
import time

def generate_qwen_explanation(disease, confidence, behavior, level="clinician"):
    """Generate dual-level explanation using Qwen2.5-3B-Instruct"""
    
    regions = behavior.get("anatomical_regions", [])
    spatial = behavior.get("spatial_analysis", {})
    pattern = "focal abnormality" if spatial.get("is_focal") else "diffuse pattern"
    coverage = spatial.get("attention_percentage", 0)

    if regions:
        region_list = ", ".join([
            f"{r['name'].replace('_',' ')} ({r['attention_score']*100:.0f}%)"
            for r in regions[:3]
        ])
    else:
        region_list = "No specific regions highlighted"

    if level == "clinician":
        user_prompt = f"""You are an expert radiologist writing a technical report. Follow this exact structure:

**FINDING SUMMARY:**
- Primary diagnosis: {disease}
- Model confidence: {confidence*100:.1f}%
- Pattern type: {pattern}

**MODEL ATTENTION ANALYSIS:**
The AI diagnostic model focused on: {region_list}
Total coverage area: {coverage:.1f}% of the thoracic cavity

**CLINICAL INTERPRETATION:**
Provide a detailed technical analysis (100-120 words) explaining:
1. What this finding means anatomically
2. Why the model focused on these specific regions
3. The radiological significance of the {pattern}

**DIFFERENTIAL CONSIDERATIONS:**
List 2-3 differential diagnoses and key distinguishing features.

Use medical terminology appropriate for radiologists."""

    else:  # patient
        user_prompt = f"""You are a caring doctor explaining X-ray results to a patient. Write in simple, clear language.

The X-ray shows signs of {disease.lower()} affecting {region_list.lower()}.

Please explain this to the patient in 120-150 words:
1. What it means in simple terms
2. What the AI system detected
3. What happens next (treatment/next steps)
4. End with a supportive, calming statement

Guidelines:
- Write at 8th-grade reading level
- Avoid medical jargon
- Use "you" and "your"
- Be warm, supportive, and honest
Do NOT include confidence percentages."""

    messages = [
        {"role": "system", "content": "You are an expert medical AI assistant specializing in radiology."},
        {"role": "user",   "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to('cuda')

    t0 = time.time()
    with torch.no_grad():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.92,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_time = time.time() - t0

    # Qwen returns input+output tokens — slice off the input
    output_ids = generated_ids[0][len(inputs.input_ids[0]):]
    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return response, gen_time

print("Qwen2.5 explanation generator ready!")
print("Clinician level : 4-section structured radiological report")
print("Patient level   : 8th-grade friendly explanation")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 3: NLP] — Generate & Display Explanations
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("COMPONENT 3: NLP — GENERATING DUAL-LEVEL EXPLANATIONS")
print("=" * 70)

print(f"\nGenerating CLINICIAN explanation...")
clinician_exp, c_time = generate_qwen_explanation(
    results['disease'], results['confidence'], results['behavior'], "clinician"
)
print(f"   Done in {c_time:.2f}s  ({len(clinician_exp.split())} words)")

print(f"\nGenerating PATIENT explanation...")
patient_exp, p_time = generate_qwen_explanation(
    results['disease'], results['confidence'], results['behavior'], "patient"
)
print(f"   Done in {p_time:.2f}s  ({len(patient_exp.split())} words)")

# Store
results['clinician_exp'] = clinician_exp
results['patient_exp']   = patient_exp

print("\n" + "=" * 70)
print("CLINICIAN-LEVEL EXPLANATION")
print("=" * 70)
print(clinician_exp)

print("\n" + "=" * 70)
print("PATIENT-LEVEL EXPLANATION")
print("=" * 70)
print(patient_exp)

print("\n" + "=" * 70)
print("FULL PIPELINE COMPLETE")
print("   CV → DenseNet-121")
print("   XAI → Grad-CAM  | Faithfulness 0.6353")
print(f"  NLP → Qwen2.5-3B | ~{(c_time+p_time):.1f}s total")
print("=" * 70)
